# 03 — Bronze layer: immutable raw-to-Parquet ingestion

**Goal:** convert each of the nine source CSV files into a Bronze Parquet dataset while preserving source columns and source-level values. We add only ingestion metadata; we do not deduplicate, impute, filter or standardize business fields in Bronze.

**Prerequisites:** run `01_data_ingestion.ipynb` successfully. The source CSVs must exist in Google Drive.

## 1. Mount Drive and start Spark

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT = Path("/content/drive/MyDrive/ecommerce-data-engineering")
RAW = PROJECT / "data" / "raw"
BRONZE = PROJECT / "data" / "bronze"
REPORTS = PROJECT / "reports"
BRONZE.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("olist-bronze-ingestion")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

assert len(list(RAW.glob("*.csv"))) == 9, "Run notebook 01 first."
print("Spark:", spark.version)

## 2. Define Bronze ingestion rules

- Read source columns as strings to avoid premature type coercion.
- Keep every source row, including duplicates and nulls.
- Add `_source_file`, `_ingested_at_utc` and `_bronze_table`.
- Store each table in its own Parquet directory.
- Keep original CSV files unchanged in `data/raw/`.

The raw CSV files remain the immutable source of truth. Bronze Parquet is a query-friendly representation of those source records.

In [ ]:
source_files = sorted(RAW.glob("*.csv"))
bronze_audit = []

for source_path in source_files:
    table_name = source_path.stem
    output_path = BRONZE / table_name

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(str(source_path))
    )

    bronze_df = (
        source_df
        .withColumn("_source_file", F.lit(source_path.name))
        .withColumn("_ingested_at_utc", F.current_timestamp())
        .withColumn("_bronze_table", F.lit(table_name))
    )

    # Coalesce limits small-file creation for this modest portfolio dataset.
    # It does not remove or deduplicate records.
    bronze_df.coalesce(2).write.mode("overwrite").parquet(str(output_path))

    written_df = spark.read.parquet(str(output_path))
    source_count = source_df.count()
    bronze_count = written_df.count()

    if source_count != bronze_count:
        raise RuntimeError(
            f"Record-count mismatch for {table_name}: "
            f"source={source_count}, bronze={bronze_count}"
        )

    bronze_audit.append({
        "table_name": table_name,
        "source_file": source_path.name,
        "source_records": source_count,
        "bronze_records": bronze_count,
        "source_columns": len(source_df.columns),
        "bronze_columns": len(written_df.columns),
        "output_path": str(output_path),
        "status": "SUCCESS"
    })

    print(f"{table_name}: {source_count:,} rows -> Bronze OK")

## 3. Review the Bronze ingestion audit

In [ ]:
import pandas as pd

bronze_audit_df = pd.DataFrame(bronze_audit)
display(bronze_audit_df)

audit_path = REPORTS / "bronze_ingestion_audit.csv"
bronze_audit_df.to_csv(audit_path, index=False)

total_source = int(bronze_audit_df["source_records"].sum())
total_bronze = int(bronze_audit_df["bronze_records"].sum())

print(f"Total source records: {total_source:,}")
print(f"Total Bronze records: {total_bronze:,}")
print("All tables preserved:", total_source == total_bronze)
print("Audit saved to:", audit_path)

assert total_source == total_bronze
assert (bronze_audit_df["status"] == "SUCCESS").all()

## 4. Inspect a Bronze table and verify metadata

In [ ]:
bronze_orders = spark.read.parquet(str(BRONZE / "olist_orders_dataset"))
bronze_orders.printSchema()
bronze_orders.show(5, truncate=False)

print("Bronze order rows:", bronze_orders.count())
print("Bronze columns:", bronze_orders.columns)

metadata_check = bronze_orders.select(
    "_source_file", "_ingested_at_utc", "_bronze_table"
).distinct()
metadata_check.show(truncate=False)

## 5. Bronze layer validation checklist

- [ ] Nine Bronze table directories exist.
- [ ] Every source row count matches its Bronze row count.
- [ ] The three ingestion metadata columns are present.
- [ ] Original CSV files remain unchanged.
- [ ] `reports/bronze_ingestion_audit.csv` exists.

**Important:** duplicates and missing values are intentionally retained in Bronze. We will handle them in Silver using explicit, documented business rules.

**Next notebook:** `04_silver_layer.ipynb` — data type standardization, duplicate investigation, null handling, referential checks and feature engineering.